# Bed Utilization & Occupancy Visualization

### Milestone 3 – Member 2

*Objective:*
To visualize the bed utilization and occupancy findings identified during the analysis of admissions and bed capacity data.

*Data Sources:*
- Admissions (cleaned)
- Departments (cleaned)
- Bed Capacity (benchmark dataset)

*Visualization Tool:* Plotly

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import plot

### Libraries Used

- *Pandas:* Used for loading and handling the cleaned datasets.
- *Plotly:* Used to create interactive visualizations.
- *Graph Objects:* Used for KPI cards and the final dashboard layout.

In [2]:
admissions = pd.read_csv("../data/processed/admissions_clean.csv")
departments = pd.read_csv("../data/processed/departments_clean.csv")
bed_capacity = pd.read_csv("../data/raw/bed_capacity.csv")

print("Admissions Shape:", admissions.shape)
print("Departments Shape:", departments.shape)
print("Bed Capacity Shape:", bed_capacity.shape)

Admissions Shape: (5000, 8)
Departments Shape: (20, 2)
Bed Capacity Shape: (20, 3)


In [3]:
display(admissions.head())
display(departments.head())
display(bed_capacity.head())

,Admission_ID,Patient_ID,Doctor_ID,Department_ID,Admission_Date,Discharge_Date,Status,Length_of_Stay_Days
0,A00001,P00001,DR00186,D002,2025-10-20,2025-10-28,Critical,8
1,A00002,P00002,DR00281,D019,2025-01-21,2025-01-23,Recovered,2
2,A00003,P00003,DR00105,D020,2025-06-24,2025-07-03,Discharged,9
3,A00004,P00004,DR00312,D004,2025-07-23,2025-07-25,Critical,2
4,A00005,P00005,DR00430,D018,2025-04-06,2025-04-07,Under Treatment,1


,department_id,department_name
0,D001,Cardiology
1,D002,Neurology
2,D003,Orthopedics
3,D004,Pediatrics
4,D005,Oncology


,department_id,department_name,Total_Beds
0,D001,Cardiology,40
1,D002,Neurology,30
2,D003,Orthopedics,35
3,D004,Pediatrics,45
4,D005,Oncology,35


### Dataset Loading

The cleaned Admissions and Departments datasets, along with the Bed Capacity benchmark dataset prepared by
Member 1 (Type A), are used for visualization.

No additional cleaning is performed here. The purpose of this notebook is to visualize the bed utilization and
occupancy findings from the existing analysis.

In [4]:
print("Admissions columns:")
print(admissions.columns.tolist())

print("\nBed Capacity columns:")
print(bed_capacity.columns.tolist())

Admissions columns:
['Admission_ID', 'Patient_ID', 'Doctor_ID', 'Department_ID', 'Admission_Date', 'Discharge_Date', 'Status', 'Length_of_Stay_Days']

Bed Capacity columns:
['department_id', 'department_name', 'Total_Beds']


## 1. Building Department-wise Daily & Monthly Occupancy

### Business Question
*What is the bed occupancy rate and available-bed count per department?*

Following Member 1's approach: for every day in the admissions period, a patient is counted as occupying a bed
in their department if `Admission_Date <= day < Discharge_Date`. These daily counts are then averaged by month
and department to get a stable, time-aware occupancy figure (rather than a raw admissions-to-beds ratio, which
would overstate occupancy since patients turn over beds throughout the year).

In [5]:
admissions["Admission_Date"] = pd.to_datetime(admissions["Admission_Date"])
admissions["Discharge_Date"] = pd.to_datetime(admissions["Discharge_Date"])

date_range = pd.date_range(
    start=admissions["Admission_Date"].min(),
    end=admissions["Discharge_Date"].max(),
    freq="D"
)

daily_occupancy = []

for date in date_range:
    active_patients = admissions[
        (admissions["Admission_Date"] <= date) &
        (admissions["Discharge_Date"] > date)
    ]

    occupied = (
        active_patients
        .groupby("Department_ID")
        .size()
        .reset_index(name="Occupied_Beds")
    )

    occupied["Date"] = date
    daily_occupancy.append(occupied)

daily_occupancy = pd.concat(daily_occupancy, ignore_index=True)
daily_occupancy["Month"] = daily_occupancy["Date"].dt.to_period("M")

print("Daily occupancy records:", daily_occupancy.shape)
display(daily_occupancy.head())

Daily occupancy records: (6010, 4)


,Department_ID,Occupied_Beds,Date,Month
0,D002,3,2025-01-01,2025-01
1,D003,1,2025-01-01,2025-01
2,D004,1,2025-01-01,2025-01
3,D005,1,2025-01-01,2025-01
4,D006,1,2025-01-01,2025-01


### Note on the Monthly Trend Calculation

Member 1's original monthly trend divided the mean occupied beds (averaged only over departments active that
month) by the **Total_Beds summed across all 20 departments**. This mismatched numerator/denominator produced
misleadingly low figures (1% peak, 0% lowest). The calculation below fixes this by computing each department's
own occupancy rate first, and only then averaging rates across departments — so every month is compared on the
same basis.

In [6]:
monthly_occupied = (
    daily_occupancy
    .groupby(["Month", "Department_ID"])["Occupied_Beds"]
    .mean()
    .reset_index(name="Average_Occupied_Beds")
)

monthly_bed_analysis = monthly_occupied.merge(
    bed_capacity,
    left_on="Department_ID",
    right_on="department_id",
    how="left"
)

monthly_bed_analysis["Available_Beds"] = (
    monthly_bed_analysis["Total_Beds"] - monthly_bed_analysis["Average_Occupied_Beds"]
)

monthly_bed_analysis["Occupancy_Rate"] = (
    monthly_bed_analysis["Average_Occupied_Beds"] / monthly_bed_analysis["Total_Beds"]
) * 100

monthly_bed_analysis["Average_Occupied_Beds"] = monthly_bed_analysis["Average_Occupied_Beds"].round().astype(int)
monthly_bed_analysis["Available_Beds"] = monthly_bed_analysis["Available_Beds"].round().astype(int)

display(monthly_bed_analysis.head())

,Month,Department_ID,Average_Occupied_Beds,department_id,department_name,Total_Beds,Available_Beds,Occupancy_Rate
0,2025-01,D001,3,D001,Cardiology,40,37,7.142857
1,2025-01,D002,2,D002,Neurology,30,28,7.471264
2,2025-01,D003,8,D003,Orthopedics,35,27,23.870968
3,2025-01,D004,6,D004,Pediatrics,45,39,13.763441
4,2025-01,D005,9,D005,Oncology,35,26,24.884793


## 2. Department-wise Occupancy Rate

### Business Question
*Which departments are overloaded, near capacity, or underutilized?*

Departments are classified using the same thresholds Member 1 used for the EDA: **Under 40% = Underutilized**,
**40–79% = Near Capacity**, **80%+ = Over Capacity**.

In [7]:
department_summary = (
    monthly_bed_analysis
    .groupby(["Department_ID", "department_name"])
    .agg(
        Total_Beds=("Total_Beds", "first"),
        Avg_Occupancy_Rate=("Occupancy_Rate", "mean"),
        Avg_Available_Beds=("Available_Beds", "mean")
    )
    .reset_index()
)

def capacity_status(rate):
    if rate >= 80:
        return "Over Capacity"
    elif rate >= 40:
        return "Near Capacity"
    else:
        return "Underutilized"

department_summary["Capacity_Status"] = department_summary["Avg_Occupancy_Rate"].apply(capacity_status)

department_summary["Avg_Occupancy_Rate"] = department_summary["Avg_Occupancy_Rate"].round(1)
department_summary["Avg_Available_Beds"] = department_summary["Avg_Available_Beds"].round(1)

department_summary = department_summary.sort_values("Avg_Occupancy_Rate", ascending=False)

display(department_summary)

,Department_ID,department_name,Total_Beds,Avg_Occupancy_Rate,Avg_Available_Beds,Capacity_Status
15,D016,Radiology,10,46.1,5.4,Near Capacity
19,D020,Dental,10,43.8,5.7,Near Capacity
17,D018,Ophthalmology,12,38.3,7.4,Underutilized
6,D007,Dermatology,10,34.1,6.6,Underutilized
18,D019,Endocrinology,18,31.2,12.5,Underutilized
5,D006,ENT,15,24.7,11.4,Underutilized
12,D013,Pulmonology,25,22.0,19.7,Underutilized
11,D012,Psychiatry,25,17.9,20.5,Underutilized
9,D010,Nephrology,25,17.2,20.7,Underutilized
13,D014,Gastroenterology,25,17.0,20.8,Underutilized


In [8]:
status_colors = {
    "Over Capacity": "#d62728",
    "Near Capacity": "#ff7f0e",
    "Underutilized": "#1f77b4"
}

plot_data = department_summary.sort_values("Avg_Occupancy_Rate", ascending=True)

fig_occupancy = px.bar(
    plot_data,
    x="Avg_Occupancy_Rate",
    y="department_name",
    orientation="h",
    color="Capacity_Status",
    color_discrete_map=status_colors,
    title="Department-wise Bed Occupancy Rate",
    text="Avg_Occupancy_Rate"
)

fig_occupancy.add_vline(x=40, line_dash="dash", line_color="gray")
fig_occupancy.add_vline(x=80, line_dash="dash", line_color="gray")

fig_occupancy.update_traces(
    texttemplate="%{text}%",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Occupancy Rate: %{x:.1f}%<extra></extra>"
)

fig_occupancy.update_layout(
    title_x=0.5,
    xaxis_title="Average Occupancy Rate (%)",
    yaxis_title="Department",
    height=520,
    margin=dict(l=140, r=80, t=50, b=50),
    legend_title_text="Capacity Status"
)

fig_occupancy.show()

### Key Finding

Radiology (46%) and Dental (43%) run closest to capacity, while General Surgery (8%), ICU (9%) and Emergency (10%)
show the lowest average occupancy. No department currently crosses the 80% over-capacity line.

## 3. Available Beds per Department

### Business Question
*How many beds are available, on average, in each department?*

In [9]:
avail_data = department_summary.sort_values("Avg_Available_Beds", ascending=True)

fig_available = px.bar(
    avail_data,
    x="Avg_Available_Beds",
    y="department_name",
    orientation="h",
    title="Average Available Beds by Department",
    text="Avg_Available_Beds",
    color_discrete_sequence=["#2ca02c"]
)

fig_available.update_traces(
    texttemplate="%{text}",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Available Beds: %{x:.1f}<extra></extra>"
)

fig_available.update_layout(
    title_x=0.5,
    xaxis_title="Average Available Beds",
    yaxis_title="Department",
    height=520,
    margin=dict(l=140, r=80, t=50, b=50)
)

fig_available.show()

### Key Finding

Emergency (60 beds) and ICU (50 beds) carry the largest average number of open beds, consistent with their low
occupancy rates above — both departments are provisioned for surge capacity rather than routine steady-state load.

## 4. Monthly Occupancy Trend (Corrected)

### Business Question
*How does overall hospital-wide bed occupancy change over the year?*

In [10]:
monthly_overall = (
    monthly_bed_analysis
    .groupby("Month")["Occupancy_Rate"]
    .mean()
    .reset_index()
)

monthly_overall["Month"] = monthly_overall["Month"].astype(str)

peak_month = monthly_overall.loc[monthly_overall["Occupancy_Rate"].idxmax()]
lowest_month = monthly_overall.loc[monthly_overall["Occupancy_Rate"].idxmin()]

print("Peak Occupancy Month:", peak_month["Month"], "-", round(peak_month["Occupancy_Rate"], 1), "%")
print("Lowest Occupancy Month:", lowest_month["Month"], "-", round(lowest_month["Occupancy_Rate"], 1), "%")

Peak Occupancy Month: 2025-06 - 22.9 %
Lowest Occupancy Month: 2025-11 - 8.4 %


In [11]:
fig_trend = px.line(
    monthly_overall,
    x="Month",
    y="Occupancy_Rate",
    markers=True,
    title="Monthly Overall Bed Occupancy Trend"
)

fig_trend.update_traces(
    hovertemplate="<b>%{x}</b><br>Occupancy Rate: %{y:.1f}%<extra></extra>"
)

fig_trend.update_layout(
    title_x=0.5,
    xaxis_title="Month",
    yaxis_title="Average Occupancy Rate (%)",
    height=300,
    margin=dict(l=60, r=40, t=50, b=50)
)

fig_trend.show()

## 5. Bed Utilization KPI Cards

The following KPI cards summarize the overall bed utilization position identified during the analysis.

In [12]:
overall_occupancy = monthly_bed_analysis["Occupancy_Rate"].mean()
most_utilized = department_summary.iloc[0]
least_utilized = department_summary.iloc[-1]
avg_available_beds = department_summary["Avg_Available_Beds"].mean()

print(f"Average Occupancy Rate: {overall_occupancy:.0f}%")
print(f"Most Utilized Department: {most_utilized['department_name']} ({most_utilized['Avg_Occupancy_Rate']:.0f}%)")
print(f"Least Utilized Department: {least_utilized['department_name']} ({least_utilized['Avg_Occupancy_Rate']:.0f}%)")
print(f"Average Available Beds: {avg_available_beds:.0f}")

Average Occupancy Rate: 21%
Most Utilized Department: Radiology (46%)
Least Utilized Department: General Surgery (8%)
Average Available Beds: 24


In [13]:
fig_kpi = make_subplots(
    rows=1,
    cols=4,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    horizontal_spacing=0.05
)

fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(overall_occupancy),
        title={"text": "<b>AVG OCCUPANCY RATE</b>", "font": {"size": 14}},
        number={"suffix": "%", "valueformat": ".0f", "font": {"size": 26}}
    ),
    row=1, col=1
)

fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(most_utilized["Avg_Occupancy_Rate"]),
        title={"text": f"<b>MOST UTILIZED</b><br>{most_utilized['department_name']}", "font": {"size": 13}},
        number={"suffix": "%", "valueformat": ".0f", "font": {"size": 26}}
    ),
    row=1, col=2
)

fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(least_utilized["Avg_Occupancy_Rate"]),
        title={"text": f"<b>LEAST UTILIZED</b><br>{least_utilized['department_name']}", "font": {"size": 13}},
        number={"suffix": "%", "valueformat": ".0f", "font": {"size": 26}}
    ),
    row=1, col=3
)

fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(avg_available_beds),
        title={"text": "<b>AVG AVAILABLE BEDS</b>", "font": {"size": 14}},
        number={"valueformat": ".0f", "font": {"size": 26}}
    ),
    row=1, col=4
)

fig_kpi.update_layout(
    height=190,
    margin=dict(l=10, r=10, t=50, b=10),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

fig_kpi.show()

## 6. Capacity Status Distribution

### Business Question
*How many departments fall into each capacity category?*

In [14]:
status_counts = department_summary["Capacity_Status"].value_counts().reset_index()
status_counts.columns = ["Status", "Count"]

fig_status = px.pie(
    status_counts,
    names="Status",
    values="Count",
    hole=0.55,
    title="Department Capacity Status Distribution",
    color="Status",
    color_discrete_map=status_colors
)

fig_status.update_traces(
    textinfo="label+value",
    hovertemplate="<b>%{label}</b><br>Departments: %{value}<extra></extra>"
)

fig_status.update_layout(
    title_x=0.5,
    height=300,
    margin=dict(l=40, r=20, t=50, b=40)
)

fig_status.show()

# Final Bed Utilization & Occupancy Dashboard

The following dashboard combines the visualizations created above into a single interactive page.

In [15]:
html_kpi = plot(fig_kpi, output_type="div", include_plotlyjs="cdn")
html_occupancy = plot(fig_occupancy, output_type="div", include_plotlyjs=False)
html_available = plot(fig_available, output_type="div", include_plotlyjs=False)
html_trend = plot(fig_trend, output_type="div", include_plotlyjs=False)
html_status = plot(fig_status, output_type="div", include_plotlyjs=False)

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Bed Utilization & Occupancy Dashboard</title>
    <style>
        * {{
            box-sizing: border-box;
        }}
        body {{
            margin: 0;
            padding: 10px;
            font-family: Arial, sans-serif;
            background: #f5f6f8;
        }}
        .dashboard {{
            width: 100%;
            max-width: 1400px;
            margin: auto;
        }}
        .header {{
            text-align: center;
            margin-bottom: 8px;
        }}
        .header h1 {{
            margin: 3px;
            font-size: 24px;
        }}
        .header p {{
            margin: 3px;
            font-size: 13px;
        }}
        .kpi {{
            background: white;
            border-radius: 8px;
            margin-bottom: 8px;
            width: 100%;
            height: 190px;
            padding: 0;
            overflow: hidden;
        }}
        .kpi .plotly-graph-div {{
            width: 100% !important;
            height: 190px !important;
        }}
        .charts {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 8px;
            width: 100%;
        }}
        .chart {{
            background: white;
            border-radius: 8px;
            width: 100%;
            height: 350px;
            padding: 2px;
            overflow: visible;
        }}
        .chart.tall {{
            height: 550px;
        }}
        .chart .plotly-graph-div {{
            width: 100% !important;
            height: 100% !important;
        }}
        @media (max-width: 900px) {{
            .header h1 {{
                font-size: 20px;
            }}
            .charts {{
                grid-template-columns: 1fr;
            }}
        }}
    </style>
</head>
<body>
<div class="dashboard">

    <div class="header">
        <h1>Bed Utilization & Occupancy Dashboard</h1>
        <p>Department-wise Occupancy, Available Beds, and Capacity Analysis</p>
    </div>

    <div class="kpi">
        {html_kpi}
    </div>

    <div class="charts">
        <div class="chart tall">
            {html_occupancy}
        </div>
        <div class="chart tall">
            {html_available}
        </div>
        <div class="chart">
            {html_trend}
        </div>
        <div class="chart">
            {html_status}
        </div>
    </div>

</div>
</body>
</html>
"""

with open("../reports/bed_utilization_occupancy_dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("Dashboard created successfully!")

Dashboard created successfully!
